# Séance 4 — Convolution et filtrage

**Niveau :** L2 Mathématiques — **Durée :** 3 h en présentiel + environ 2 h de travail personnel.

**Thème :** Noyaux, convolution discrète, flou, contours et effets de bord.

Les images sont représentées par des tableaux NumPy : niveaux de gris `(hauteur, largeur)` ou couleurs `(hauteur, largeur, 3)`.

## Objectifs et déroulé

### Objectifs
- Comprendre un noyau comme opérateur local.
- Implémenter une convolution 2D simple.
- Relier lissage, bruit et détection de contours.

### Déroulé indicatif (3 h)
1. 30 min — voisinage et convolution/corrélation.
2. 40 min — noyaux moyenneur, gaussien et Sobel.
3. 40 min — exercices.
4. 45 min — étape projet : prétraitement et analyse de l’impact du filtrage.
5. 25 min — effets de bord et coût calculatoire.

**Travail personnel (~2 h)** : tester trois noyaux et expliquer leur effet sur les détails fins.

## Rappel — Convolution discrète

**Convolution, filtrage, corrélation.** 

La convolution généralise l'idée de moyenne glissante.
Elle est la représentation mathématique de la notion de filtre linéaire en traitement du signal. 
La convlution s'applique aussi bien à des données temporelles (en traitement du signal par exemple) qu'à des données spatiales (en traitement d'image). 

La convolution discrète est une somme pondérée sur un voisinage de valeurs. 
Cette pondération est définie par une (petite) matrice K, le _noyau_ de (la) convolution. 
Une convolution appliquée à une image I combine les pixels voisins au moyen du noyau K :

$
(I*K)(y,x)=\sum_{i=-r}^{r}\sum_{j=-r}^{r}K(i,j)I(y-i,x-j).
$

Attention : on distinguera le produit de convolution ci-dessus d'un produit matriciel. 

**Compréhension.** Pour K de taille 3x3, représenter la relation précédente par un schéma. Il est d'usage de représenter le centre du noyau 3x3 par $K(0,0)$ et $r=1$. Rappel : les indices de ligne croissent vers le bas, et ceux de colonnes vers la droite.

Un filtre local appliquée à une image effectue un traitement identique.

En statistique, la corrélation croisée utilise une formule très voisine. (source wikipedia) 
Ainsi certaines bibliothèques calculent en fait une corrélation (sans retourner le noyau). 
Pour un noyau symétrique, les deux opérations coïncident.
Pour un noyau non symétrique, l’orientation peut changer.

La convolution ou le filtrage d'une image de taille $H \times W$ par noyau de taille $k \times k$ ($k=2*r+1$) coûte  $O(HW k^2)$.

**Exemple.**

Un filtre moyenneur 3×3 est
$
K=\frac19\begin{pmatrix}1&1&1\\1&1&1\\1&1&1\end{pmatrix}.
$

Il atténue certaines variations locales, mais floute aussi les détails. 
Un noyau dont la somme vaut 1 préserve généralement les zones constantes loin des bords.

Le _gradient_ d’une image, considérée comme une fonction I(x,y), est  

$
G(I) =(\partial I/\partial x,\partial I/\partial y).$

Ce façon discrète, ces dérivées partielles s'approchent, par exemple au premier ordre, comme la différence des valeurs de deux pixels voisins horizontalement et verticalement.

Sa norme vaut
$
\|G\|=\sqrt{G_x^2+G_y^2}.
$

Le gradient d'une image mesure la variation locale de l'intensité de ses pixels.
Les contours et les changements importants de valeurs correspondent souvent à une norme élevée du gradient. 

Les noyaux du filtre de Sobel approchent aussi les dérivées horizontale et verticale. 
Ils fournissent un opérateur simple et efficace de détection de contours dans une image.

**Attention au format de calcul.** 
Les dérivées peuvent par exemple être négatives : éviter de les transformer trop tôt en valeurs entières. 
Au bord, une règle de prolongement est nécessaire. 

**Dans le notebook :** tester un noyau sur une image constante ; regarder les valeurs avant conversion en entiers ; vérifier la convention convolution/corrélation.

**Premiers liens :** 

- [Traitement d’image](https://fr.wikipedia.org/wiki/Traitement_d%27image)
- [Convolution](https://fr.wikipedia.org/wiki/Convolution) 
- [Gradient](https://fr.wikipedia.org/wiki/Gradient)
- [Filtre de Sobel](https://fr.wikipedia.org/wiki/Filtre_de_Sobel)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

## Outil de convolution

On implémente une corrélation 2D avec remplissage par zéros (_zero padding_). 

Rappel : pour les noyaux symétriques, corrélation et convolution donnent le même résultat.

In [1]:
def correlation2d(img, noyau):
    img = np.asarray(img, dtype=float)
    k = np.asarray(noyau, dtype=float)
    if k.ndim != 2 or any(n % 2 == 0 for n in k.shape):
        raise ValueError("Le noyau doit être 2D et de dimensions impaires")
    py, px = k.shape[0]//2, k.shape[1]//2
    pad = np.pad(img, ((py,py),(px,px)), mode="constant")
    out = np.zeros_like(img, dtype=float)
    for y in range(img.shape[0]):
        for x in range(img.shape[1]):
            out[y,x] = np.sum(pad[y:y+k.shape[0], x:x+k.shape[1]] * k)
    return out

**Petit travail préliminaire.**

1. Comprendre le traitement de cette fonction. Que représentent `px`et `py`? pourquoi définir `pad` ?   
1. Expliciter l'expression mathématique calculée par cette fonction. Comment se compare-t-elle à l'expression de la convolution indiqueé plus haut ? Que faudrait-il vérifier pour assurer que cette fonction calcule aussi la convolution de `noyau` appliquée à `img`?   
2. Compléter cette fonction avec :
    - les annotations de types des paramètres,
    - un _docstring_ contenant les commentaires explicatifs de la fonction.

\# **Ecrire votre réponse ici**

## Exercice 1 — moyenneur

**Énoncé.** Appliquer un noyau 3 × 3 dont les coefficients valent 1/9 à une image constante. Que doit-on obtenir loin des bords ?

In [2]:
# Ecrire votre code réponse ici**

\# **Ecrire votre réponse ici**

## Exercice 2 — gradient

**Énoncé.** 

1. Définir une image partagée en deux parties noire et blanche, par exemple horizontalement ou verticalement ou selon une diagonale.
2. Appliquer sur ces images, les noyaux de Sobel pour calculer les gradients horizontaux et verticaux correspondants ainsi que la norme du gradient.
3. Afficher les résultats obtenus et commentez-les. 

In [2]:
# Ecrire votre code réponse ici**

\# **Ecrire votre réponse ici**

## Étape projet

**Rappel : vous pouvez regrouper vos livrables successifs dans un notebook unique.**

**Livrable séance 4 :** un prétraitement configurable (choix d'un filtrage/noyau de convolution, par exemple un filtre moyenneur) et une comparaison quantitative et visuelle avant/après filtrage ainsi que la visualisation de la norme du gradient.

**Indications.**

- Une fois comprise, la fonction `mse()` suivante pourra être utile à la comparaison quantitative.

In [3]:
def mse(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    if a.shape != b.shape:
        raise ValueError("Les deux images doivent avoir la même forme")
    return float(np.mean((a-b)**2))